In [ ]:
# Importe
from pathlib import Path
import json
import platform
import warnings
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
try:
    import pm4py
except ImportError as e:
    raise ImportError('PM4Py fehlt. Bitte in der .venv installieren: pip install -U pm4py') from e
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, balanced_accuracy_score, accuracy_score, brier_score_loss, log_loss, confusion_matrix, roc_curve, precision_recall_curve
print('Notebook 05 läuft.')
print('Python:', platform.python_version())
print('Platform:', platform.platform())


In [ ]:
# Pfade und Einstellungen
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data_raw'
LOG_PATH = DATA_RAW / 'BPI_Challenge_2018.xes.gz'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'prediction_design_baseline_modeling'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for d in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
N_JOBS = -1
PRIMARY_LABEL = 'label_scd_p90_or_global'
ROBUSTNESS_LABELS_FOR_LIGHT_MODELING = ['label_scd_p90_or_yearnorm', 'label_scd_p90_or_no_inspection_global', 'label_scd_p90_or_no_inspection_yearnorm', 'label_path_change_or_objection', 'label_temporal_duration_p90_yearnorm']
PREFIX_LENGTHS_MAIN = [0, 5, 10, 20]
REQUIRE_MIN_EVENTS_FOR_PREFIX = True
TOP_N_RAW_ACTIVITIES = 30
TOP_N_COMBINED_ACTIVITIES = 60
TOP_N_RESOURCES = 30
MIN_CASES_PER_CATEGORY = 50
RUN_RANDOM_FOREST = True
RUN_DECISION_TREE = True
RUN_LOGISTIC_REGRESSION = True
RUN_DUMMY_BASELINE = True
RUN_LIGHT_ROBUSTNESS_TARGET_MODELS = True
ROBUSTNESS_PREFIX_LENGTH = 10
ROBUSTNESS_MODELS = ['logreg_balanced', 'rf_balanced']
SPLIT_STRATEGY = 'auto_chronological'
if not LOG_PATH.exists():
    candidates = sorted(DATA_RAW.glob('*.xes*')) + sorted(DATA_RAW.glob('**/*.xes*'))
    print('Gefundene XES-Kandidaten:')
    for c in candidates[:20]:
        print('-', c)
    if candidates:
        LOG_PATH = candidates[0]
        print('Nutze automatisch:', LOG_PATH)
    else:
        raise FileNotFoundError(f'Keine XES/XES.GZ-Datei in {DATA_RAW} gefunden.')
print('Project root:', PROJECT_ROOT)
print('Log path:', LOG_PATH)
print('Output root:', OUTPUT_ROOT)


In [ ]:
# Hilfsfunktionen
created_tables = []
created_figures = []
analysis_notes = []

def save_csv(obj, filename, index=True):
    path = TABLE_DIR / filename
    if isinstance(obj, pd.Series):
        obj.to_frame().to_csv(path, index=index, encoding='utf-8-sig')
    else:
        obj.to_csv(path, index=index, encoding='utf-8-sig')
    created_tables.append(path)
    return path

def save_json(obj, filename):
    path = TABLE_DIR / filename
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    created_tables.append(path)
    return path

def save_fig(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    created_figures.append(path)
    return path

def q(series, quantile):
    return float(pd.to_numeric(series, errors='coerce').quantile(quantile))

def robust_to_bool(s, default=False):
    if isinstance(s, pd.Series):
        if s.dtype == bool:
            return s.fillna(default).astype(bool)
        s_str = s.astype(str).str.strip().str.lower()
        true_values = {'true', '1', '1.0', 'yes', 'y', 'ja', 'wahr', 't'}
        false_values = {'false', '0', '0.0', 'no', 'n', 'nein', 'falsch', 'nan', 'none', '<na>', '', 'f'}
        out = pd.Series(bool(default), index=s.index)
        out[s_str.isin(true_values)] = True
        out[s_str.isin(false_values)] = False
        numeric = pd.to_numeric(s, errors='coerce')
        out[numeric.fillna(0) > 0] = True
        return out.astype(bool)
    if pd.isna(s):
        return bool(default)
    if isinstance(s, str):
        return s.strip().lower() in {'true', '1', '1.0', 'yes', 'y', 'ja', 'wahr', 't'}
    return bool(s)

def ensure_label_bools(df, label_cols=None):
    if label_cols is None:
        label_cols = [c for c in df.columns if str(c).startswith('label_')]
    actual = [c for c in label_cols if c in df.columns]
    for col in actual:
        df[col] = robust_to_bool(df[col])
    return actual

def numeric_series(df_, col, default=0):
    if col in df_.columns:
        return pd.to_numeric(df_[col], errors='coerce').fillna(default)
    return pd.Series(default, index=df_.index)

def safe_auc(metric_func, y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(metric_func(y_true, y_score))
    except Exception:
        return np.nan

def safe_log_loss(y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(log_loss(y_true, y_score, labels=[0, 1]))
    except Exception:
        return np.nan

def select_threshold_by_f1(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return (0.5, np.nan)
    if np.nanmax(y_score) == np.nanmin(y_score):
        return (float(y_score[0]) if len(y_score) else 0.5, np.nan)
    thresholds = np.unique(np.quantile(y_score, np.linspace(0.01, 0.99, 99)))
    best_t = 0.5
    best_f1 = -1
    for t in thresholds:
        pred = (y_score >= t).astype(int)
        score = f1_score(y_true, pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_t = float(t)
    return (best_t, float(best_f1))

def compute_binary_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)
    if len(y_true) == 0:
        return {'n': 0, 'positive': 0, 'prevalence_pct': np.nan, 'threshold': float(threshold), 'roc_auc': np.nan, 'pr_auc_average_precision': np.nan, 'brier_score': np.nan, 'log_loss': np.nan, 'accuracy': np.nan, 'balanced_accuracy': np.nan, 'precision': np.nan, 'recall': np.nan, 'f1': np.nan, 'specificity': np.nan, 'false_positive_rate': np.nan, 'false_negative_rate': np.nan, 'tn': 0, 'fp': 0, 'fn': 0, 'tp': 0}
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if tn + fp > 0 else np.nan
    fpr = fp / (tn + fp) if tn + fp > 0 else np.nan
    fnr = fn / (tp + fn) if tp + fn > 0 else np.nan
    return {'n': int(len(y_true)), 'positive': int(y_true.sum()), 'prevalence_pct': float(y_true.mean() * 100) if len(y_true) else np.nan, 'threshold': float(threshold), 'roc_auc': safe_auc(roc_auc_score, y_true, y_score), 'pr_auc_average_precision': safe_auc(average_precision_score, y_true, y_score), 'brier_score': float(brier_score_loss(y_true, y_score)) if len(np.unique(y_true)) >= 2 else np.nan, 'log_loss': safe_log_loss(y_true, y_score), 'accuracy': float(accuracy_score(y_true, y_pred)), 'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)) if len(np.unique(y_true)) >= 2 else np.nan, 'precision': float(precision_score(y_true, y_pred, zero_division=0)), 'recall': float(recall_score(y_true, y_pred, zero_division=0)), 'f1': float(f1_score(y_true, y_pred, zero_division=0)), 'specificity': float(specificity) if not pd.isna(specificity) else np.nan, 'false_positive_rate': float(fpr) if not pd.isna(fpr) else np.nan, 'false_negative_rate': float(fnr) if not pd.isna(fnr) else np.nan, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=True)

def reduce_rare_categories(series, min_count=MIN_CASES_PER_CATEGORY, other_label='__OTHER__'):
    s = series.astype(str).fillna('__MISSING__')
    counts = s.value_counts(dropna=False)
    keep = set(counts[counts >= min_count].index)
    return s.where(s.isin(keep), other_label)

def get_feature_names_from_preprocessor(preprocessor):
    names = []
    for name, transformer, cols in preprocessor.transformers_:
        if name == 'remainder' and transformer == 'drop':
            continue
        if transformer == 'drop':
            continue
        if hasattr(transformer, 'named_steps'):
            last_step = list(transformer.named_steps.values())[-1]
            if hasattr(last_step, 'get_feature_names_out'):
                try:
                    step_names = last_step.get_feature_names_out(cols)
                    names.extend(step_names)
                except Exception:
                    names.extend(cols)
            else:
                names.extend(cols)
        else:
            names.extend(cols)
    return list(map(str, names))
print('Helper bereit. Robuste Boolean-Konvertierung aktiv.')


In [ ]:
# Log laden
print('Lade XES-Log ...')
log = pm4py.read_xes(str(LOG_PATH))
event_df = pm4py.convert_to_dataframe(log)
print('Event DataFrame:', event_df.shape)
CASE_COL = 'case:concept:name' if 'case:concept:name' in event_df.columns else None
ACTIVITY_COL = 'concept:name' if 'concept:name' in event_df.columns else None
RAW_ACTIVITY_COL = 'activity' if 'activity' in event_df.columns else ACTIVITY_COL
TIME_COL = 'time:timestamp' if 'time:timestamp' in event_df.columns else None
ORDER_COL = 'identity:id' if 'identity:id' in event_df.columns else 'eventid' if 'eventid' in event_df.columns else None
if CASE_COL is None or ACTIVITY_COL is None or TIME_COL is None:
    raise RuntimeError(f'Kernspalten fehlen: CASE_COL={CASE_COL}, ACTIVITY_COL={ACTIVITY_COL}, TIME_COL={TIME_COL}')
event_df[CASE_COL] = event_df[CASE_COL].astype(str)
event_df[TIME_COL] = pd.to_datetime(event_df[TIME_COL], errors='coerce')
for c in ['doctype', 'subprocess', RAW_ACTIVITY_COL]:
    if c not in event_df.columns:
        event_df[c] = '__missing__'
event_df['combined_activity'] = event_df['doctype'].astype(str) + ' | ' + event_df['subprocess'].astype(str) + ' | ' + event_df[RAW_ACTIVITY_COL].astype(str)
sort_cols = [CASE_COL, TIME_COL]
if ORDER_COL is not None:
    sort_cols.append(ORDER_COL)
else:
    sort_cols.append(ACTIVITY_COL)
event_df = event_df.sort_values(sort_cols, kind='mergesort').reset_index(drop=True)
event_df['_event_pos_in_case'] = event_df.groupby(CASE_COL).cumcount() + 1
basic_info = {'events': int(len(event_df)), 'cases': int(event_df[CASE_COL].nunique()), 'activities': int(event_df[ACTIVITY_COL].nunique()), 'raw_activities': int(event_df[RAW_ACTIVITY_COL].nunique()), 'combined_activities': int(event_df['combined_activity'].nunique()), 'columns': int(event_df.shape[1]), 'timestamp_min': str(event_df[TIME_COL].min()), 'timestamp_max': str(event_df[TIME_COL].max()), 'case_col': CASE_COL, 'activity_col': ACTIVITY_COL, 'raw_activity_col': RAW_ACTIVITY_COL, 'time_col': TIME_COL, 'order_col': ORDER_COL}
save_json(basic_info, '00_basic_info.json')
print(json.dumps(basic_info, indent=2, ensure_ascii=False))
event_df.head()


In [ ]:
# Fallmerkmale und Labels
PREVIOUS_LABEL_CORE = PROJECT_ROOT / 'outputs' / 'label_robustness_feature_availability' / 'tables' / '17_case_level_label_robustness_core.csv'
case_times = event_df.groupby(CASE_COL)[TIME_COL].agg(case_start='min', case_end='max')
case_times['duration_days'] = (case_times['case_end'] - case_times['case_start']).dt.total_seconds() / (3600 * 24)
case_event_count = event_df.groupby(CASE_COL).size().rename('event_count')
combined_counts = event_df.groupby([CASE_COL, 'combined_activity']).size().reset_index(name='count')
combined_rework = combined_counts[combined_counts['count'] > 1].groupby(CASE_COL)['count'].apply(lambda x: int((x - 1).sum())).rename('combined_rework_extra')
inspection_mask = event_df['combined_activity'].astype(str).str.lower().str.contains('inspection', regex=False) | event_df['subprocess'].astype(str).str.lower().str.contains('inspection', regex=False) | event_df['doctype'].astype(str).str.lower().str.contains('inspection', regex=False)
inspection_by_case = event_df.assign(_is_inspection=inspection_mask).groupby(CASE_COL)['_is_inspection'].agg(has_inspection_context='any', inspection_event_count='sum')
no_insp_df = event_df.loc[~inspection_mask].copy()
event_count_no_insp = no_insp_df.groupby(CASE_COL).size().rename('event_count_no_inspection')
if len(no_insp_df) > 0:
    cc_no_insp = no_insp_df.groupby([CASE_COL, 'combined_activity']).size().reset_index(name='count')
    combined_rework_no_insp = cc_no_insp[cc_no_insp['count'] > 1].groupby(CASE_COL)['count'].apply(lambda x: int((x - 1).sum())).rename('combined_rework_extra_no_inspection')
else:
    combined_rework_no_insp = pd.Series(dtype=float, name='combined_rework_extra_no_inspection')
first_events = event_df.groupby(CASE_COL).head(1).copy().set_index(CASE_COL)
case_context = pd.DataFrame(index=case_times.index)
case_context['case_start_month'] = case_times['case_start'].dt.month
case_context['case_start_quarter'] = case_times['case_start'].dt.quarter
case_context['case_start_weekday'] = case_times['case_start'].dt.weekday
case_context['case_start_year_from_timestamp'] = case_times['case_start'].dt.year.astype('Int64').astype(str)
for src_col, target_col in [('case:year', 'case:year'), ('case:department', 'case:department'), ('doctype', 'first_doctype'), ('subprocess', 'first_subprocess'), (RAW_ACTIVITY_COL, 'first_activity'), ('combined_activity', 'first_combined_activity'), ('org:resource', 'first_resource')]:
    if src_col in first_events.columns:
        case_context[target_col] = first_events[src_col].astype(str)
if 'case:year' not in case_context.columns:
    case_context['case:year'] = case_context['case_start_year_from_timestamp']
if 'case:department' not in case_context.columns:
    case_context['case:department'] = 'unknown'
case_context['n_doctypes_full_case'] = event_df.groupby(CASE_COL)['doctype'].nunique() if 'doctype' in event_df.columns else np.nan
case_context['n_subprocesses_full_case'] = event_df.groupby(CASE_COL)['subprocess'].nunique() if 'subprocess' in event_df.columns else np.nan
case_context['n_resources_full_case'] = event_df.groupby(CASE_COL)['org:resource'].nunique() if 'org:resource' in event_df.columns else np.nan
case_df_recomputed = case_times.join(case_event_count).join(combined_rework).join(inspection_by_case).join(event_count_no_insp).join(combined_rework_no_insp).join(case_context).reset_index()
for col in ['combined_rework_extra', 'inspection_event_count', 'event_count_no_inspection', 'combined_rework_extra_no_inspection', 'event_count', 'duration_days']:
    if col in case_df_recomputed.columns:
        case_df_recomputed[col] = pd.to_numeric(case_df_recomputed[col], errors='coerce').fillna(0)
case_df_recomputed['has_inspection_context'] = robust_to_bool(case_df_recomputed.get('has_inspection_context', pd.Series(False, index=case_df_recomputed.index)))
if PREVIOUS_LABEL_CORE.exists():
    print('Lade Case-Level-Labels aus Schritt 04:', PREVIOUS_LABEL_CORE)
    previous_core = pd.read_csv(PREVIOUS_LABEL_CORE)
    previous_core[CASE_COL] = previous_core[CASE_COL].astype(str)
    label_cols_prev = [c for c in previous_core.columns if c.startswith('label_')]
    for c in label_cols_prev:
        previous_core[c] = robust_to_bool(previous_core[c])
    context_cols_prev = [c for c in ['case:year', 'case:department'] if c in previous_core.columns]
    merge_cols = [CASE_COL] + label_cols_prev + context_cols_prev
    case_df = case_df_recomputed.copy()
    case_df[CASE_COL] = case_df[CASE_COL].astype(str)
    case_df = case_df.merge(previous_core[merge_cols], on=CASE_COL, how='left', suffixes=('', '_prev'))
    for c in ['case:year', 'case:department']:
        prev_c = f'{c}_prev'
        if prev_c in case_df.columns:
            case_df[c] = case_df[c].where(case_df[c].notna(), case_df[prev_c])
            case_df = case_df.drop(columns=[prev_c])
else:
    print('Kein vorheriger Label-Core gefunden. Rekonstruiere zentrale Labels.')
    case_df = case_df_recomputed.copy()
    case_df[CASE_COL] = case_df[CASE_COL].astype(str)
for col in ['event_count', 'combined_rework_extra', 'duration_days', 'event_count_no_inspection', 'combined_rework_extra_no_inspection']:
    if col not in case_df.columns:
        case_df[col] = 0
    case_df[col] = pd.to_numeric(case_df[col], errors='coerce').fillna(0)
thresholds = {'event_count_p90_global': q(case_df['event_count'], 0.9), 'event_count_p95_global': q(case_df['event_count'], 0.95), 'combined_rework_p90_global': q(case_df['combined_rework_extra'], 0.9), 'combined_rework_p95_global': q(case_df['combined_rework_extra'], 0.95), 'duration_p90_global': q(case_df['duration_days'], 0.9), 'duration_p95_global': q(case_df['duration_days'], 0.95), 'event_count_no_inspection_p90_global': q(case_df['event_count_no_inspection'], 0.9), 'combined_rework_no_inspection_p90_global': q(case_df['combined_rework_extra_no_inspection'], 0.9)}
case_df['label_scd_p90_or_global'] = (case_df['event_count'] >= thresholds['event_count_p90_global']) | (case_df['combined_rework_extra'] >= thresholds['combined_rework_p90_global'])
case_df['label_scd_p95_or_global'] = (case_df['event_count'] >= thresholds['event_count_p95_global']) | (case_df['combined_rework_extra'] >= thresholds['combined_rework_p95_global'])
case_df['label_scd_p90_and_global'] = (case_df['event_count'] >= thresholds['event_count_p90_global']) & (case_df['combined_rework_extra'] >= thresholds['combined_rework_p90_global'])
case_df['label_scd_p90_or_no_inspection_global'] = (case_df['event_count_no_inspection'] >= thresholds['event_count_no_inspection_p90_global']) | (case_df['combined_rework_extra_no_inspection'] >= thresholds['combined_rework_no_inspection_p90_global'])
case_df['label_temporal_duration_p90_global'] = case_df['duration_days'] >= thresholds['duration_p90_global']
case_df['case:year'] = case_df.get('case:year', case_df.get('case_start_year_from_timestamp', 'unknown')).astype(str).fillna('unknown')
for metric in ['event_count', 'combined_rework_extra', 'duration_days', 'event_count_no_inspection', 'combined_rework_extra_no_inspection']:
    case_df[f'_thr_{metric}_p90_by_year'] = case_df.groupby('case:year')[metric].transform(lambda s: pd.to_numeric(s, errors='coerce').quantile(0.9))
case_df['label_scd_p90_or_yearnorm'] = (case_df['event_count'] >= case_df['_thr_event_count_p90_by_year']) | (case_df['combined_rework_extra'] >= case_df['_thr_combined_rework_extra_p90_by_year'])
case_df['label_scd_p90_or_no_inspection_yearnorm'] = (case_df['event_count_no_inspection'] >= case_df['_thr_event_count_no_inspection_p90_by_year']) | (case_df['combined_rework_extra_no_inspection'] >= case_df['_thr_combined_rework_extra_no_inspection_p90_by_year'])
case_df['label_temporal_duration_p90_yearnorm'] = case_df['duration_days'] >= case_df['_thr_duration_days_p90_by_year']
for old_col in ['label_path_change_or_objection', 'label_path_change_or_objection_x', 'label_path_change_or_objection_y']:
    if old_col in case_df.columns:
        case_df = case_df.drop(columns=[old_col])
if 'subprocess' in event_df.columns:
    change_obj = event_df['subprocess'].astype(str).str.lower().isin(['change', 'objection'])
    tmp = event_df.assign(_change_obj=change_obj).groupby(CASE_COL)['_change_obj'].any().rename('label_path_change_or_objection').reset_index()
    tmp[CASE_COL] = tmp[CASE_COL].astype(str)
    case_df = case_df.merge(tmp, on=CASE_COL, how='left')
    case_df['label_path_change_or_objection'] = robust_to_bool(case_df['label_path_change_or_objection'])
else:
    case_df['label_path_change_or_objection'] = False
for old_col in ['label_remove_document', 'label_remove_document_x', 'label_remove_document_y']:
    if old_col in case_df.columns:
        case_df = case_df.drop(columns=[old_col])
if RAW_ACTIVITY_COL in event_df.columns:
    remove_doc = event_df[RAW_ACTIVITY_COL].astype(str).str.lower().eq('remove document')
    tmp = event_df.assign(_remove_doc=remove_doc).groupby(CASE_COL)['_remove_doc'].any().rename('label_remove_document').reset_index()
    tmp[CASE_COL] = tmp[CASE_COL].astype(str)
    case_df = case_df.merge(tmp, on=CASE_COL, how='left')
    case_df['label_remove_document'] = robust_to_bool(case_df['label_remove_document'])
else:
    case_df['label_remove_document'] = False
LABEL_COLS_ALL = ensure_label_bools(case_df)
label_diag_rows = []
for c in LABEL_COLS_ALL:
    label_diag_rows.append({'label': c, 'positive_cases': int(case_df[c].sum()), 'total_cases': int(len(case_df)), 'share_positive_pct': float(case_df[c].mean() * 100), 'dtype': str(case_df[c].dtype)})
label_diag_df = pd.DataFrame(label_diag_rows).sort_values('share_positive_pct', ascending=False)
save_json(thresholds, '01_label_thresholds_modeling.json')
save_csv(case_df.head(50), '01_case_df_preview_first50.csv', index=False)
save_csv(label_diag_df, '01b_label_boolean_diagnosis.csv', index=False)
if PRIMARY_LABEL not in case_df.columns:
    raise RuntimeError(f'PRIMARY_LABEL fehlt nach Label-Konstruktion: {PRIMARY_LABEL}')
if case_df[PRIMARY_LABEL].nunique() < 2:
    raise RuntimeError(f'PRIMARY_LABEL hat nur eine Klasse. positives={int(case_df[PRIMARY_LABEL].sum())}, total={len(case_df)}. Bitte Label-Rekonstruktion prüfen.')
print('Case DF:', case_df.shape)
print('Primary positives:', int(case_df[PRIMARY_LABEL].sum()), f'({case_df[PRIMARY_LABEL].mean() * 100:.2f}%)')
print('Labels:', LABEL_COLS_ALL)
case_df[[CASE_COL, 'case:year', 'case:department', 'event_count', 'combined_rework_extra', PRIMARY_LABEL]].head()


In [ ]:
# Labelprävalenz und Leakage
if PRIMARY_LABEL not in case_df.columns:
    raise RuntimeError(f'PRIMARY_LABEL fehlt: {PRIMARY_LABEL}')
LABEL_COLS_ALL = ensure_label_bools(case_df)
label_list_for_modeling = [PRIMARY_LABEL] + [c for c in ROBUSTNESS_LABELS_FOR_LIGHT_MODELING if c in case_df.columns and c != PRIMARY_LABEL]
label_list_for_modeling = [c for c in label_list_for_modeling if c in case_df.columns]
for col in label_list_for_modeling:
    case_df[col] = robust_to_bool(case_df[col])
label_prevalence = []
for col in label_list_for_modeling:
    label_prevalence.append({'label': col, 'cases_positive': int(case_df[col].sum()), 'cases_total': int(len(case_df)), 'share_positive_pct': float(case_df[col].mean() * 100), 'nunique': int(case_df[col].nunique())})
label_prevalence_df = pd.DataFrame(label_prevalence).sort_values('share_positive_pct', ascending=False)
save_csv(label_prevalence_df, '02_label_prevalence_for_modeling.csv', index=False)
if case_df[PRIMARY_LABEL].nunique() < 2:
    raise RuntimeError(f'PRIMARY_LABEL ist für Modeling ungeeignet: nur eine Klasse. Positives={int(case_df[PRIMARY_LABEL].sum())}, total={len(case_df)}')
feature_policy_rows = [{'feature_or_group': 'duration_days', 'decision': 'exclude', 'reason': 'Vollständige Falllaufzeit; erst nach Fallende bekannt.'}, {'feature_or_group': 'event_count', 'decision': 'exclude_as_feature', 'reason': 'Bestandteil des SCD-Targets; vollständige Fallinformation.'}, {'feature_or_group': 'combined_rework_extra', 'decision': 'exclude_as_feature', 'reason': 'Bestandteil des SCD-Targets; vollständige Fallinformation.'}, {'feature_or_group': 'event_count_no_inspection', 'decision': 'exclude_as_feature', 'reason': 'Vollständige Fallinformation; nur für Robustheitslabel genutzt.'}, {'feature_or_group': 'label_*', 'decision': 'exclude', 'reason': 'Targets/Outcome-Labels dürfen niemals als Features genutzt werden.'}, {'feature_or_group': 'case_end', 'decision': 'exclude', 'reason': 'Endzeitpunkt ist zum frühen Vorhersagezeitpunkt nicht verfügbar.'}, {'feature_or_group': 'full-case n_doctypes/n_subprocesses/n_resources', 'decision': 'exclude', 'reason': 'Full-case Proxy; erst am Fallende vollständig bekannt.'}, {'feature_or_group': 'case:year / case:department', 'decision': 'allow_with_note', 'reason': 'Statische Kontextattribute; nur falls fachlich zum Prozessstart bekannt.'}, {'feature_or_group': 'start calendar features', 'decision': 'allow', 'reason': 'Startzeit-Kontext ohne Outcome-Information.'}, {'feature_or_group': 'first event categorical features', 'decision': 'allow', 'reason': 'Zum Start/ersten beobachteten Event verfügbar.'}, {'feature_or_group': 'prefix counts first k events', 'decision': 'allow', 'reason': 'Nur Informationen aus tatsächlich beobachtetem Prefix.'}]
feature_policy_df = pd.DataFrame(feature_policy_rows)
save_csv(feature_policy_df, '03_feature_leakage_policy.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = label_prevalence_df.sort_values('share_positive_pct', ascending=True)
ax.barh(plot_df['label'], plot_df['share_positive_pct'])
ax.set_xlabel('Anteil positiver Fälle (%)')
ax.set_title('Label-Prävalenz für Modeling')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_01_label_prevalence_modeling.png')
label_prevalence_df


In [ ]:
# Chronologischer Split
def make_chronological_split(case_df_in, target_col, strategy='auto_chronological'):
    df = case_df_in.copy()
    df[target_col] = robust_to_bool(df[target_col])
    df['split'] = None
    year_values = pd.Series(df['case:year'].astype(str).unique()).sort_values().tolist() if 'case:year' in df.columns else []
    usable_years = [y for y in year_values if y not in ['nan', 'None', 'unknown', '<NA>']]
    if strategy == 'auto_chronological' and len(usable_years) >= 3:
        try:
            usable_years_sorted = sorted(usable_years, key=lambda x: int(float(x)))
        except Exception:
            usable_years_sorted = sorted(usable_years)
        test_year = usable_years_sorted[-1]
        val_year = usable_years_sorted[-2]
        train_years = set(usable_years_sorted[:-2])
        df.loc[df['case:year'].astype(str).isin(train_years), 'split'] = 'train'
        df.loc[df['case:year'].astype(str).eq(str(val_year)), 'split'] = 'validation'
        df.loc[df['case:year'].astype(str).eq(str(test_year)), 'split'] = 'test'
        df.loc[df['split'].isna(), 'split'] = 'train'
        split_basis = f'case:year chronological; train={sorted(train_years)}, validation={val_year}, test={test_year}'
    else:
        tmp = df.sort_values('case_start').reset_index(drop=True)
        n = len(tmp)
        train_end = int(n * 0.6)
        val_end = int(n * 0.8)
        tmp.loc[:train_end - 1, 'split'] = 'train'
        tmp.loc[train_end:val_end - 1, 'split'] = 'validation'
        tmp.loc[val_end:, 'split'] = 'test'
        df = tmp
        split_basis = 'case_start_time quantile split 60/20/20'
    split_summary = []
    for split in ['train', 'validation', 'test']:
        sub = df[df['split'] == split]
        split_summary.append({'split': split, 'n_cases': int(len(sub)), 'positive_cases': int(sub[target_col].sum()) if len(sub) else 0, 'negative_cases': int((~sub[target_col]).sum()) if len(sub) else 0, 'prevalence_pct': float(sub[target_col].mean() * 100) if len(sub) else np.nan, 'target_nunique': int(sub[target_col].nunique()) if len(sub) else 0, 'case_start_min': str(sub['case_start'].min()) if len(sub) else None, 'case_start_max': str(sub['case_start'].max()) if len(sub) else None, 'case_year_values': ', '.join(map(str, sorted(sub['case:year'].astype(str).unique()))) if 'case:year' in sub.columns and len(sub) else ''})
    split_summary_df = pd.DataFrame(split_summary)
    invalid = False
    for split in ['train', 'validation', 'test']:
        sub = df[df['split'] == split]
        if len(sub) == 0 or sub[target_col].nunique() < 2:
            invalid = True
    if invalid and split_basis.startswith('case:year'):
        print('Warnung: Year-Split hat eine Klasse in mindestens einem Split nicht abgedeckt. Fallback auf time quantile.')
        return make_chronological_split(case_df_in, target_col, strategy='time_quantile')
    return (df[[CASE_COL, 'split']], split_summary_df, split_basis)
split_df, split_summary_df, split_basis = make_chronological_split(case_df, PRIMARY_LABEL, SPLIT_STRATEGY)
case_df = case_df.drop(columns=['split'], errors='ignore').merge(split_df, on=CASE_COL, how='left')
case_df[PRIMARY_LABEL] = robust_to_bool(case_df[PRIMARY_LABEL])
save_csv(split_summary_df, '04_split_summary_primary.csv', index=False)
save_json({'split_basis': split_basis}, '04_split_basis.json')
print('Split basis:', split_basis)
split_summary_df


In [ ]:
# Split abbilden
fig, ax = plt.subplots(figsize=(8, 5))
split_order = ['train', 'validation', 'test']
plot_df = split_summary_df.set_index('split').loc[split_order].reset_index()
ax.bar(plot_df['split'], plot_df['n_cases'])
ax.set_ylabel('Anzahl Cases')
ax.set_title('Chronologischer Modeling-Split')
ax.grid(axis='y', alpha=0.3)
save_fig(fig, 'fig_02_split_case_counts.png')
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(plot_df['split'], plot_df['prevalence_pct'])
ax.set_ylabel('Positive Fälle (%)')
ax.set_title('Target-Prävalenz je Split')
ax.grid(axis='y', alpha=0.3)
save_fig(fig, 'fig_03_split_target_prevalence.png')


In [ ]:
# Merkmalsvokabular
train_case_ids = set(case_df.loc[case_df['split'] == 'train', CASE_COL].astype(str))
train_events = event_df[event_df[CASE_COL].isin(train_case_ids)].copy()
TOP_RAW_ACTIVITIES = train_events[RAW_ACTIVITY_COL].astype(str).value_counts().head(TOP_N_RAW_ACTIVITIES).index.tolist()
TOP_COMBINED_ACTIVITIES = train_events['combined_activity'].astype(str).value_counts().head(TOP_N_COMBINED_ACTIVITIES).index.tolist()
TOP_RESOURCES = train_events['org:resource'].astype(str).value_counts().head(TOP_N_RESOURCES).index.tolist() if 'org:resource' in train_events.columns else []
TOP_DOCTYPES = train_events['doctype'].astype(str).value_counts().index.tolist()
TOP_SUBPROCESSES = train_events['subprocess'].astype(str).value_counts().index.tolist()
vocab_info = {'top_raw_activities_n': len(TOP_RAW_ACTIVITIES), 'top_combined_activities_n': len(TOP_COMBINED_ACTIVITIES), 'top_resources_n': len(TOP_RESOURCES), 'top_doctypes_n': len(TOP_DOCTYPES), 'top_subprocesses_n': len(TOP_SUBPROCESSES), 'top_raw_activities': TOP_RAW_ACTIVITIES, 'top_combined_activities': TOP_COMBINED_ACTIVITIES, 'top_resources': TOP_RESOURCES, 'top_doctypes': TOP_DOCTYPES, 'top_subprocesses': TOP_SUBPROCESSES}
save_json(vocab_info, '05_feature_vocabularies_train_only.json')
pd.DataFrame({'top_raw_activities': pd.Series(TOP_RAW_ACTIVITIES), 'top_combined_activities': pd.Series(TOP_COMBINED_ACTIVITIES), 'top_resources': pd.Series(TOP_RESOURCES)}).head(20)


In [ ]:
# Präfixmerkmale
def make_case_static_features(case_df_in):
    static = case_df_in[[CASE_COL, 'case_start', 'case:year', 'case:department', 'split', PRIMARY_LABEL]].copy()
    for lab in ROBUSTNESS_LABELS_FOR_LIGHT_MODELING:
        if lab in case_df_in.columns and lab not in static.columns:
            static[lab] = case_df_in[lab].values
    static['start_month'] = pd.to_datetime(static['case_start'], errors='coerce').dt.month
    static['start_quarter'] = pd.to_datetime(static['case_start'], errors='coerce').dt.quarter
    static['start_weekday'] = pd.to_datetime(static['case_start'], errors='coerce').dt.weekday
    static['start_year_from_timestamp'] = pd.to_datetime(static['case_start'], errors='coerce').dt.year.astype('Int64').astype(str)
    for col in ['first_doctype', 'first_subprocess', 'first_activity', 'first_combined_activity', 'first_resource']:
        if col in case_df_in.columns:
            static[col] = case_df_in[col].astype(str).values
        else:
            static[col] = '__missing__'
    for col in ['case:year', 'case:department', 'first_doctype', 'first_subprocess', 'first_activity', 'first_combined_activity', 'first_resource', 'start_year_from_timestamp']:
        if col in static.columns:
            static[col] = reduce_rare_categories(static[col], min_count=MIN_CASES_PER_CATEGORY)
    return static

def make_prefix_event_features(event_df_in, prefix_len):
    if prefix_len == 0:
        base = pd.DataFrame({CASE_COL: case_df[CASE_COL].astype(str).values})
        base['prefix_len'] = 0
        base['prefix_event_count'] = 0
        base['prefix_duration_hours'] = 0.0
        base['prefix_n_raw_activities'] = 0
        base['prefix_n_combined_activities'] = 0
        base['prefix_n_doctypes'] = 0
        base['prefix_n_subprocesses'] = 0
        base['prefix_n_resources'] = 0
        base['prefix_raw_rework_extra'] = 0
        base['prefix_combined_rework_extra'] = 0
        base['prefix_has_change_or_objection'] = False
        base['prefix_has_inspection_context'] = False
        base['prefix_has_remove_document'] = False
        base['prefix_has_payment_activity'] = False
        base['prefix_has_decision_activity'] = False
        base['last_activity_prefix'] = '__none__'
        base['last_subprocess_prefix'] = '__none__'
        base['last_doctype_prefix'] = '__none__'
        base['last_resource_prefix'] = '__none__'
        return base
    prefix_events = event_df_in[event_df_in['_event_pos_in_case'] <= prefix_len].copy()
    agg = prefix_events.groupby(CASE_COL).agg(prefix_event_count=(ACTIVITY_COL, 'size'), prefix_start=(TIME_COL, 'min'), prefix_end=(TIME_COL, 'max'), prefix_n_raw_activities=(RAW_ACTIVITY_COL, 'nunique'), prefix_n_combined_activities=('combined_activity', 'nunique'), prefix_n_doctypes=('doctype', 'nunique'), prefix_n_subprocesses=('subprocess', 'nunique'))
    if 'org:resource' in prefix_events.columns:
        n_res = prefix_events.groupby(CASE_COL)['org:resource'].nunique().rename('prefix_n_resources')
        agg = agg.join(n_res)
    else:
        agg['prefix_n_resources'] = 0
    agg['prefix_duration_hours'] = (agg['prefix_end'] - agg['prefix_start']).dt.total_seconds() / 3600
    agg = agg.drop(columns=['prefix_start', 'prefix_end'])
    raw_counts = prefix_events.groupby([CASE_COL, RAW_ACTIVITY_COL]).size().reset_index(name='count')
    raw_rework = raw_counts[raw_counts['count'] > 1].groupby(CASE_COL)['count'].apply(lambda x: int((x - 1).sum())).rename('prefix_raw_rework_extra')
    combined_counts = prefix_events.groupby([CASE_COL, 'combined_activity']).size().reset_index(name='count')
    combined_rework = combined_counts[combined_counts['count'] > 1].groupby(CASE_COL)['count'].apply(lambda x: int((x - 1).sum())).rename('prefix_combined_rework_extra')
    agg = agg.join(raw_rework).join(combined_rework)
    agg['prefix_raw_rework_extra'] = agg['prefix_raw_rework_extra'].fillna(0)
    agg['prefix_combined_rework_extra'] = agg['prefix_combined_rework_extra'].fillna(0)
    lower_sub = prefix_events['subprocess'].astype(str).str.lower()
    lower_act = prefix_events[RAW_ACTIVITY_COL].astype(str).str.lower()
    lower_comb = prefix_events['combined_activity'].astype(str).str.lower()
    flags = pd.DataFrame({CASE_COL: prefix_events[CASE_COL].values})
    flags['prefix_has_change_or_objection'] = lower_sub.isin(['change', 'objection']).values
    flags['prefix_has_inspection_context'] = lower_comb.str.contains('inspection', regex=False).values
    flags['prefix_has_remove_document'] = lower_act.eq('remove document').values
    flags['prefix_has_payment_activity'] = lower_comb.str.contains('payment', regex=False).values
    flags['prefix_has_decision_activity'] = lower_act.str.contains('decide', regex=False).values
    flags_agg = flags.groupby(CASE_COL).any()
    agg = agg.join(flags_agg)
    last_events = prefix_events.groupby(CASE_COL).tail(1).set_index(CASE_COL)
    agg['last_activity_prefix'] = last_events[RAW_ACTIVITY_COL].astype(str)
    agg['last_subprocess_prefix'] = last_events['subprocess'].astype(str)
    agg['last_doctype_prefix'] = last_events['doctype'].astype(str)
    agg['last_resource_prefix'] = last_events['org:resource'].astype(str) if 'org:resource' in last_events.columns else '__missing__'
    for val in TOP_RAW_ACTIVITIES:
        col = f"cnt_raw_activity__{str(val).replace(' ', '_').replace('/', '_')[:80]}"
        tmp = prefix_events[RAW_ACTIVITY_COL].astype(str).eq(str(val)).groupby(prefix_events[CASE_COL]).sum().rename(col)
        agg = agg.join(tmp)
    for val in TOP_COMBINED_ACTIVITIES:
        safe_val = str(val).replace(' ', '_').replace('/', '_').replace('|', '_')[:90]
        col = f'cnt_combined__{safe_val}'
        tmp = prefix_events['combined_activity'].astype(str).eq(str(val)).groupby(prefix_events[CASE_COL]).sum().rename(col)
        agg = agg.join(tmp)
    for val in TOP_DOCTYPES:
        col = f"cnt_doctype__{str(val).replace(' ', '_').replace('/', '_')[:80]}"
        tmp = prefix_events['doctype'].astype(str).eq(str(val)).groupby(prefix_events[CASE_COL]).sum().rename(col)
        agg = agg.join(tmp)
    for val in TOP_SUBPROCESSES:
        col = f"cnt_subprocess__{str(val).replace(' ', '_').replace('/', '_')[:80]}"
        tmp = prefix_events['subprocess'].astype(str).eq(str(val)).groupby(prefix_events[CASE_COL]).sum().rename(col)
        agg = agg.join(tmp)
    if 'org:resource' in prefix_events.columns:
        for val in TOP_RESOURCES:
            col = f"cnt_resource__{str(val).replace(' ', '_').replace('/', '_')[:80]}"
            tmp = prefix_events['org:resource'].astype(str).eq(str(val)).groupby(prefix_events[CASE_COL]).sum().rename(col)
            agg = agg.join(tmp)
    agg = agg.fillna(0).reset_index()
    agg['prefix_len'] = prefix_len
    for col in ['last_activity_prefix', 'last_subprocess_prefix', 'last_doctype_prefix', 'last_resource_prefix']:
        if col in agg.columns:
            agg[col] = reduce_rare_categories(agg[col], min_count=MIN_CASES_PER_CATEGORY)
    return agg

def build_modeling_dataset(prefix_len):
    static = make_case_static_features(case_df)
    prefix = make_prefix_event_features(event_df, prefix_len)
    model_df = static.merge(prefix, on=CASE_COL, how='left')
    if REQUIRE_MIN_EVENTS_FOR_PREFIX and prefix_len > 0:
        full_counts = case_df[[CASE_COL, 'event_count']].copy()
        model_df = model_df.merge(full_counts, on=CASE_COL, how='left', suffixes=('', '_full'))
        model_df['eligible_for_prefix'] = model_df['event_count'] >= prefix_len
        model_df = model_df[model_df['eligible_for_prefix']].copy()
        model_df = model_df.drop(columns=['event_count'], errors='ignore')
    else:
        model_df['eligible_for_prefix'] = True
    for c in model_df.columns:
        if c.startswith('prefix_') or c.startswith('cnt_'):
            if model_df[c].dtype == object:
                model_df[c] = model_df[c].fillna('__missing__')
            else:
                model_df[c] = pd.to_numeric(model_df[c], errors='coerce').fillna(0)
    model_df['prefix_strategy'] = f'first_{prefix_len}_events' if prefix_len > 0 else 'static_first_event'
    return model_df
example_model_df = build_modeling_dataset(5)
print('Example prefix-5 modeling dataset:', example_model_df.shape)
example_model_df.head()


In [ ]:
# Modellpipeline
def is_forced_categorical_feature(col):
    c = str(col)
    forced_exact = {'case:year', 'case:department', 'start_year_from_timestamp', 'first_doctype', 'first_subprocess', 'first_activity', 'first_combined_activity', 'first_resource', 'last_activity_prefix', 'last_subprocess_prefix', 'last_doctype_prefix', 'last_resource_prefix'}
    if c in forced_exact:
        return True
    if c.startswith('first_') or c.startswith('last_'):
        return True
    return False

def is_forced_boolean_numeric_feature(col):
    c = str(col)
    return c.startswith('prefix_has_') or c.startswith('has_') or c.startswith('is_') or c.endswith('_flag')

def identify_feature_columns(model_df, target_col):
    exclude_cols = set([CASE_COL, 'case_start', 'case_end', 'split', 'prefix_strategy', 'eligible_for_prefix', target_col])
    exclude_cols.update([c for c in model_df.columns if str(c).startswith('label_')])
    leakage_exact = {'duration_days', 'event_count', 'combined_rework_extra', 'event_count_no_inspection', 'combined_rework_extra_no_inspection', 'inspection_event_count', 'has_inspection_context', 'n_doctypes', 'n_subprocesses', 'n_resources', 'case_length', 'case_duration'}
    exclude_cols.update([c for c in model_df.columns if c in leakage_exact])
    leakage_keywords = ['case_end', '_full_case', '_thr_', 'duration_days', 'combined_rework_extra_full', 'event_count_full', 'case_duration']
    for c in model_df.columns:
        if any((k in str(c) for k in leakage_keywords)):
            exclude_cols.add(c)
    feature_cols = [c for c in model_df.columns if c not in exclude_cols]
    categorical_cols = []
    numeric_cols = []
    for c in feature_cols:
        s = model_df[c]
        if is_forced_categorical_feature(c):
            categorical_cols.append(c)
            continue
        if is_forced_boolean_numeric_feature(c):
            numeric_cols.append(c)
            continue
        if pd.api.types.is_bool_dtype(s):
            numeric_cols.append(c)
            continue
        if pd.api.types.is_numeric_dtype(s):
            numeric_cols.append(c)
            continue
        if pd.api.types.is_categorical_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s):
            non_missing = int(s.notna().sum())
            converted = pd.to_numeric(s, errors='coerce')
            convertible = int(converted.notna().sum())
            rate = convertible / non_missing if non_missing > 0 else 0
            unique_lower = set(s.dropna().astype(str).str.strip().str.lower().unique())
            boolean_like_values = {'true', 'false', '1', '0', '1.0', '0.0', 'yes', 'no', 'y', 'n', 'ja', 'nein'}
            if unique_lower and unique_lower.issubset(boolean_like_values):
                numeric_cols.append(c)
            elif rate >= 0.98 and (not is_forced_categorical_feature(c)):
                numeric_cols.append(c)
            else:
                categorical_cols.append(c)
            continue
        categorical_cols.append(c)
    numeric_cols = [c for c in feature_cols if c in set(numeric_cols) and c not in set(categorical_cols)]
    categorical_cols = [c for c in feature_cols if c in set(categorical_cols)]
    return (feature_cols, numeric_cols, categorical_cols)

def coerce_feature_types(df_in, numeric_cols, categorical_cols):
    df_out = df_in.copy()
    for c in numeric_cols:
        if c not in df_out.columns:
            continue
        if is_forced_boolean_numeric_feature(c):
            df_out[c] = robust_to_bool(df_out[c]).astype(int)
        elif pd.api.types.is_object_dtype(df_out[c]) or pd.api.types.is_string_dtype(df_out[c]):
            unique_lower = set(df_out[c].dropna().astype(str).str.strip().str.lower().unique())
            boolean_like_values = {'true', 'false', '1', '0', '1.0', '0.0', 'yes', 'no', 'y', 'n', 'ja', 'nein'}
            if unique_lower and unique_lower.issubset(boolean_like_values):
                df_out[c] = robust_to_bool(df_out[c]).astype(int)
            else:
                df_out[c] = pd.to_numeric(df_out[c], errors='coerce')
        else:
            df_out[c] = pd.to_numeric(df_out[c], errors='coerce')
    for c in categorical_cols:
        if c not in df_out.columns:
            continue
        df_out[c] = df_out[c].astype('string').fillna('__MISSING__').astype(str)
        df_out.loc[df_out[c].isin(['nan', 'None', '<NA>']), c] = '__MISSING__'
    return df_out

def feature_type_diagnostics(model_df, feature_cols, numeric_cols, categorical_cols):
    rows = []
    numeric_set = set(numeric_cols)
    categorical_set = set(categorical_cols)
    for c in feature_cols:
        s = model_df[c]
        converted = pd.to_numeric(s, errors='coerce') if c in model_df.columns else pd.Series(dtype=float)
        non_missing = int(s.notna().sum()) if c in model_df.columns else 0
        convertible = int(converted.notna().sum()) if c in model_df.columns else 0
        rows.append({'feature': c, 'assigned_type': 'numeric' if c in numeric_set else 'categorical' if c in categorical_set else 'unknown', 'dtype_original': str(s.dtype) if c in model_df.columns else 'missing', 'non_missing': non_missing, 'numeric_convertible': convertible, 'numeric_convertible_rate': convertible / non_missing if non_missing else np.nan, 'example_values': ', '.join(map(str, pd.Series(s.dropna().unique()).head(5).tolist())) if c in model_df.columns else ''})
    return pd.DataFrame(rows)

def build_preprocessor(numeric_cols, categorical_cols):
    transformers = []
    if len(numeric_cols) > 0:
        numeric_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler(with_mean=False))])
        transformers.append(('num', numeric_pipeline, numeric_cols))
    if len(categorical_cols) > 0:
        categorical_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', make_ohe())])
        transformers.append(('cat', categorical_pipeline, categorical_cols))
    if not transformers:
        raise RuntimeError('Keine Features für Modeling übrig. Leakage-Filter prüfen.')
    return ColumnTransformer(transformers=transformers, remainder='drop', sparse_threshold=0.3)

def get_model(model_name):
    if model_name == 'dummy_prior':
        return DummyClassifier(strategy='prior')
    if model_name == 'logreg_balanced':
        return LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear', random_state=RANDOM_STATE)
    if model_name == 'tree_balanced':
        return DecisionTreeClassifier(max_depth=5, min_samples_leaf=80, class_weight='balanced', random_state=RANDOM_STATE)
    if model_name == 'rf_balanced':
        return RandomForestClassifier(n_estimators=250, max_depth=None, min_samples_leaf=30, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)
    raise ValueError(f'Unbekanntes Modell: {model_name}')

def get_proba_positive(pipeline, X):
    if hasattr(pipeline, 'predict_proba'):
        proba = pipeline.predict_proba(X)
        if proba.shape[1] == 2:
            classes = getattr(pipeline, 'classes_', np.array([0, 1]))
            if len(classes) == 2:
                pos_idx = int(np.where(classes == 1)[0][0]) if 1 in classes else 1
                return proba[:, pos_idx]
            return proba[:, 1]
        if proba.shape[1] == 1:
            classes = getattr(pipeline, 'classes_', np.array([0]))
            cls = classes[0] if len(classes) else 0
            return np.ones(len(X)) if cls == 1 else np.zeros(len(X))
    if hasattr(pipeline, 'decision_function'):
        scores = pipeline.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    pred = pipeline.predict(X)
    return np.asarray(pred).astype(float)

def fit_and_evaluate(model_df, target_col, prefix_len, model_name):
    model_df = model_df.copy()
    model_df[target_col] = robust_to_bool(model_df[target_col])
    feature_cols, numeric_cols, categorical_cols = identify_feature_columns(model_df, target_col)
    if len(feature_cols) == 0:
        raise RuntimeError(f'Keine Feature-Spalten für target={target_col}, prefix={prefix_len}')
    model_df = coerce_feature_types(model_df, numeric_cols, categorical_cols)
    train_df = model_df[model_df['split'] == 'train'].copy()
    val_df = model_df[model_df['split'] == 'validation'].copy()
    test_df = model_df[model_df['split'] == 'test'].copy()
    if min(len(train_df), len(val_df), len(test_df)) == 0:
        raise RuntimeError(f'Leerer Split für target={target_col}, prefix={prefix_len}')
    if train_df[target_col].nunique() < 2:
        raise RuntimeError(f'Training enthält nur eine Klasse für target={target_col}, prefix={prefix_len}')
    X_train, y_train = (train_df[feature_cols], train_df[target_col].astype(int))
    X_val, y_val = (val_df[feature_cols], val_df[target_col].astype(int))
    X_test, y_test = (test_df[feature_cols], test_df[target_col].astype(int))
    preprocessor = build_preprocessor(numeric_cols, categorical_cols)
    model = get_model(model_name)
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    proba_train = get_proba_positive(pipeline, X_train)
    proba_val = get_proba_positive(pipeline, X_val)
    proba_test = get_proba_positive(pipeline, X_test)
    selected_threshold, val_f1_at_selected = select_threshold_by_f1(y_val, proba_val)
    rows = []
    for split_name, y_true, y_score in [('train', y_train, proba_train), ('validation', y_val, proba_val), ('test', y_test, proba_test)]:
        m_default = compute_binary_metrics(y_true, y_score, threshold=0.5)
        m_default.update({'target': target_col, 'prefix_len': prefix_len, 'prefix_strategy': f'first_{prefix_len}_events' if prefix_len > 0 else 'static_first_event', 'model': model_name, 'split': split_name, 'threshold_policy': 'fixed_0_5', 'selected_threshold_from_validation': selected_threshold, 'val_f1_at_selected_threshold': val_f1_at_selected, 'feature_count_raw': len(feature_cols), 'numeric_feature_count_raw': len(numeric_cols), 'categorical_feature_count_raw': len(categorical_cols)})
        rows.append(m_default)
        m_selected = compute_binary_metrics(y_true, y_score, threshold=selected_threshold)
        m_selected.update({'target': target_col, 'prefix_len': prefix_len, 'prefix_strategy': f'first_{prefix_len}_events' if prefix_len > 0 else 'static_first_event', 'model': model_name, 'split': split_name, 'threshold_policy': 'validation_f1_threshold', 'selected_threshold_from_validation': selected_threshold, 'val_f1_at_selected_threshold': val_f1_at_selected, 'feature_count_raw': len(feature_cols), 'numeric_feature_count_raw': len(numeric_cols), 'categorical_feature_count_raw': len(categorical_cols)})
        rows.append(m_selected)
    eval_df = pd.DataFrame(rows)
    pred_test_df = pd.DataFrame({CASE_COL: test_df[CASE_COL].values, 'target': target_col, 'prefix_len': prefix_len, 'model': model_name, 'y_true': y_test.values, 'y_score': proba_test, 'y_pred_threshold_0_5': (proba_test >= 0.5).astype(int), 'y_pred_validation_f1_threshold': (proba_test >= selected_threshold).astype(int), 'split': 'test'})
    feature_importance_df = pd.DataFrame()
    try:
        feature_names = get_feature_names_from_preprocessor(pipeline.named_steps['preprocessor'])
        fitted_model = pipeline.named_steps['model']
        if hasattr(fitted_model, 'coef_'):
            coefs = fitted_model.coef_.ravel()
            n = min(len(feature_names), len(coefs))
            feature_importance_df = pd.DataFrame({'target': target_col, 'prefix_len': prefix_len, 'model': model_name, 'feature': feature_names[:n], 'importance': coefs[:n], 'abs_importance': np.abs(coefs[:n]), 'importance_type': 'logistic_regression_coefficient'}).sort_values('abs_importance', ascending=False)
        elif hasattr(fitted_model, 'feature_importances_'):
            imps = fitted_model.feature_importances_
            n = min(len(feature_names), len(imps))
            feature_importance_df = pd.DataFrame({'target': target_col, 'prefix_len': prefix_len, 'model': model_name, 'feature': feature_names[:n], 'importance': imps[:n], 'abs_importance': np.abs(imps[:n]), 'importance_type': 'tree_feature_importance'}).sort_values('abs_importance', ascending=False)
    except Exception as e:
        analysis_notes.append(f'Feature importance extraction failed for {target_col}/{prefix_len}/{model_name}: {e}')
    curves = {}
    if len(np.unique(y_test)) >= 2:
        fpr, tpr, roc_thresholds = roc_curve(y_test, proba_test)
        prec, rec, pr_thresholds = precision_recall_curve(y_test, proba_test)
        curves['roc'] = pd.DataFrame({'fpr': fpr, 'tpr': tpr, 'threshold': roc_thresholds})
        curves['pr'] = pd.DataFrame({'precision': prec, 'recall': rec})
    calib = pd.DataFrame({'y_true': y_test.values, 'y_score': proba_test})
    try:
        if calib['y_score'].nunique() > 1:
            calib['score_bin'] = pd.qcut(calib['y_score'], q=10, duplicates='drop')
            calib_df = calib.groupby('score_bin', observed=True).agg(n=('y_true', 'size'), mean_predicted_score=('y_score', 'mean'), observed_positive_rate=('y_true', 'mean')).reset_index()
            calib_df['target'] = target_col
            calib_df['prefix_len'] = prefix_len
            calib_df['model'] = model_name
        else:
            calib_df = pd.DataFrame()
    except Exception:
        calib_df = pd.DataFrame()
    return {'pipeline': pipeline, 'eval_df': eval_df, 'pred_test_df': pred_test_df, 'feature_importance_df': feature_importance_df, 'curves': curves, 'calibration_df': calib_df, 'feature_cols': feature_cols, 'numeric_cols': numeric_cols, 'categorical_cols': categorical_cols}
print('Modeling-Funktionen bereit. v3: Featuretypen werden robust getrennt und vor Pipeline-Fit erzwungen.')


In [ ]:
# Hauptlabel modellieren
model_names = []
if RUN_DUMMY_BASELINE:
    model_names.append('dummy_prior')
if RUN_LOGISTIC_REGRESSION:
    model_names.append('logreg_balanced')
if RUN_DECISION_TREE:
    model_names.append('tree_balanced')
if RUN_RANDOM_FOREST:
    model_names.append('rf_balanced')
print('Modelle:', model_names)
print('Prefix-Längen:', PREFIX_LENGTHS_MAIN)
print('Primary target:', PRIMARY_LABEL)
all_eval = []
all_test_predictions = []
all_feature_importances = []
all_calibrations = []
all_curve_tables = []
prefix_schema_rows = []
trained_results = {}
failure_rows = []
case_df[PRIMARY_LABEL] = robust_to_bool(case_df[PRIMARY_LABEL])
if case_df[PRIMARY_LABEL].nunique() < 2:
    raise RuntimeError(f'PRIMARY_LABEL hat nur eine Klasse vor Modeling: positives={int(case_df[PRIMARY_LABEL].sum())}, total={len(case_df)}')
for prefix_len in PREFIX_LENGTHS_MAIN:
    print('\n' + '=' * 80)
    print(f'Baue Modeling Dataset für Prefix {prefix_len} ...')
    try:
        model_df = build_modeling_dataset(prefix_len)
    except Exception as e:
        msg = f'FAILED_BUILD_DATASET: prefix={prefix_len}, error={repr(e)}'
        print(msg)
        analysis_notes.append(msg)
        failure_rows.append({'stage': 'build_modeling_dataset', 'target': PRIMARY_LABEL, 'prefix_len': prefix_len, 'model': None, 'error': repr(e)})
        continue
    if PRIMARY_LABEL not in model_df.columns:
        msg = f'TARGET_MISSING: target={PRIMARY_LABEL}, prefix={prefix_len}'
        print(msg)
        analysis_notes.append(msg)
        failure_rows.append({'stage': 'target_missing', 'target': PRIMARY_LABEL, 'prefix_len': prefix_len, 'model': None, 'error': msg})
        continue
    model_df[PRIMARY_LABEL] = robust_to_bool(model_df[PRIMARY_LABEL])
    split_target_diag = []
    for sp in ['train', 'validation', 'test']:
        sub = model_df[model_df['split'] == sp]
        split_target_diag.append({'split': sp, 'n': int(len(sub)), 'positive': int(sub[PRIMARY_LABEL].sum()) if len(sub) else 0, 'negative': int((~sub[PRIMARY_LABEL]).sum()) if len(sub) else 0, 'prevalence_pct': float(sub[PRIMARY_LABEL].mean() * 100) if len(sub) else np.nan, 'nunique_target': int(sub[PRIMARY_LABEL].nunique()) if len(sub) else 0})
    diag_df = pd.DataFrame(split_target_diag)
    print('Split-/Target-Diagnose:')
    display(diag_df)
    save_csv(model_df.head(50), f'06_modeling_dataset_prefix_{prefix_len}_preview_first50.csv', index=False)
    save_csv(diag_df, f'06_split_target_diagnosis_prefix_{prefix_len}.csv', index=False)
    try:
        feature_cols, numeric_cols, categorical_cols = identify_feature_columns(model_df, PRIMARY_LABEL)
        model_df = coerce_feature_types(model_df, numeric_cols, categorical_cols)
        ftd = feature_type_diagnostics(model_df, feature_cols, numeric_cols, categorical_cols)
        save_csv(ftd, f'06_feature_type_diagnostics_prefix_{prefix_len}.csv', index=False)
    except Exception as e:
        msg = f'FAILED_IDENTIFY_FEATURES: prefix={prefix_len}, error={repr(e)}'
        print(msg)
        analysis_notes.append(msg)
        failure_rows.append({'stage': 'identify_feature_columns', 'target': PRIMARY_LABEL, 'prefix_len': prefix_len, 'model': None, 'error': repr(e)})
        continue
    prefix_schema_rows.append({'target': PRIMARY_LABEL, 'prefix_len': prefix_len, 'n_rows': int(len(model_df)), 'n_train': int((model_df['split'] == 'train').sum()), 'n_validation': int((model_df['split'] == 'validation').sum()), 'n_test': int((model_df['split'] == 'test').sum()), 'positive_share_overall_pct': float(model_df[PRIMARY_LABEL].mean() * 100), 'raw_feature_count': int(len(feature_cols)), 'numeric_feature_count': int(len(numeric_cols)), 'categorical_feature_count': int(len(categorical_cols)), 'numeric_features': ', '.join(numeric_cols[:80]), 'categorical_features': ', '.join(categorical_cols[:80])})
    print(f'Features: raw={len(feature_cols)}, numeric={len(numeric_cols)}, categorical={len(categorical_cols)}')
    invalid_split = False
    for sp in ['train', 'validation', 'test']:
        sub = model_df[model_df['split'] == sp]
        if len(sub) == 0:
            invalid_split = True
            failure_rows.append({'stage': 'split_check', 'target': PRIMARY_LABEL, 'prefix_len': prefix_len, 'model': None, 'error': f'Split {sp} ist leer.'})
        elif sub[PRIMARY_LABEL].nunique() < 2:
            msg = f'Split {sp} enthält nur eine Klasse. positive={int(sub[PRIMARY_LABEL].sum())}, n={len(sub)}'
            if sp == 'train':
                invalid_split = True
            failure_rows.append({'stage': 'split_check', 'target': PRIMARY_LABEL, 'prefix_len': prefix_len, 'model': None, 'error': msg})
    if invalid_split:
        print('WARNUNG: Training-Split ungeeignet. Prefix wird übersprungen.')
        continue
    for model_name in model_names:
        print(f'Trainiere: target={PRIMARY_LABEL}, prefix={prefix_len}, model={model_name}')
        try:
            res = fit_and_evaluate(model_df, PRIMARY_LABEL, prefix_len, model_name)
            key = (PRIMARY_LABEL, prefix_len, model_name)
            trained_results[key] = res
            all_eval.append(res['eval_df'])
            all_test_predictions.append(res['pred_test_df'])
            if len(res['feature_importance_df']):
                all_feature_importances.append(res['feature_importance_df'].head(100))
            if len(res['calibration_df']):
                all_calibrations.append(res['calibration_df'])
            for curve_name, curve_df_tmp in res['curves'].items():
                tmp = curve_df_tmp.copy()
                tmp['target'] = PRIMARY_LABEL
                tmp['prefix_len'] = prefix_len
                tmp['model'] = model_name
                tmp['curve'] = curve_name
                all_curve_tables.append(tmp)
            print('  OK')
        except Exception as e:
            msg = f'FAILED: target={PRIMARY_LABEL}, prefix={prefix_len}, model={model_name}, error={repr(e)}'
            print(' ', msg)
            analysis_notes.append(msg)
            failure_rows.append({'stage': 'fit_and_evaluate', 'target': PRIMARY_LABEL, 'prefix_len': prefix_len, 'model': model_name, 'error': repr(e)})
primary_eval_df = pd.concat(all_eval, ignore_index=True) if all_eval else pd.DataFrame()
test_predictions_df = pd.concat(all_test_predictions, ignore_index=True) if all_test_predictions else pd.DataFrame()
feature_importance_df = pd.concat(all_feature_importances, ignore_index=True) if all_feature_importances else pd.DataFrame()
calibration_df = pd.concat(all_calibrations, ignore_index=True) if all_calibrations else pd.DataFrame()
curve_df = pd.concat(all_curve_tables, ignore_index=True) if all_curve_tables else pd.DataFrame()
prefix_schema_df = pd.DataFrame(prefix_schema_rows)
failure_log_df = pd.DataFrame(failure_rows)
save_csv(primary_eval_df, '07_primary_model_evaluation_all_splits.csv', index=False)
save_csv(test_predictions_df, '08_primary_test_predictions.csv', index=False)
save_csv(feature_importance_df, '09_primary_feature_importances_top100_each_model.csv', index=False)
save_csv(calibration_df, '10_primary_calibration_bins.csv', index=False)
save_csv(curve_df, '11_primary_roc_pr_curves.csv', index=False)
save_csv(prefix_schema_df, '12_prefix_feature_schema_summary.csv', index=False)
save_csv(failure_log_df, '12b_primary_model_failure_log.csv', index=False)
print('\nFERTIG Abschnitt 9')
print('Evaluation rows:', len(primary_eval_df))
print('Failure rows:', len(failure_log_df))
if len(primary_eval_df) == 0:
    print('\nKEINE MODELLE ERFOLGREICH. Failure Log:')
    display(failure_log_df)
    analysis_notes.append('Keine Primary-Modeling-Ergebnisse; Failure Log 12b_primary_model_failure_log.csv prüfen.')
else:
    display(primary_eval_df.head())


In [ ]:
# Modellergebnisse
if len(primary_eval_df) == 0:
    msg = 'Keine Evaluationsergebnisse vorhanden. Abschnitt 9 Failure Log prüfen. Visuals werden übersprungen.'
    print(msg)
    analysis_notes.append(msg)
    best_primary_row = {}
    test_default = pd.DataFrame()
    test_selected = pd.DataFrame()
else:
    test_selected = primary_eval_df[(primary_eval_df['split'] == 'test') & (primary_eval_df['threshold_policy'] == 'validation_f1_threshold')].copy()
    test_default = primary_eval_df[(primary_eval_df['split'] == 'test') & (primary_eval_df['threshold_policy'] == 'fixed_0_5')].copy()
    if len(test_default) == 0:
        msg = 'Keine Test-Ergebnisse mit fixed_0_5 vorhanden. Primary Visuals werden teilweise übersprungen.'
        print(msg)
        analysis_notes.append(msg)
        best_primary_row = {}
    else:
        fig, ax = plt.subplots(figsize=(10, 5))
        for model_name in test_default['model'].dropna().unique():
            sub = test_default[test_default['model'] == model_name].sort_values('prefix_len')
            ax.plot(sub['prefix_len'], sub['roc_auc'], marker='o', label=model_name)
        ax.set_xlabel('Prefix-Länge')
        ax.set_ylabel('ROC-AUC auf Test')
        ax.set_title('Test ROC-AUC nach Prefix und Modell')
        ax.legend()
        ax.grid(alpha=0.3)
        save_fig(fig, 'fig_04_test_roc_auc_by_prefix_model.png')
        fig, ax = plt.subplots(figsize=(10, 5))
        for model_name in test_default['model'].dropna().unique():
            sub = test_default[test_default['model'] == model_name].sort_values('prefix_len')
            ax.plot(sub['prefix_len'], sub['pr_auc_average_precision'], marker='o', label=model_name)
        ax.set_xlabel('Prefix-Länge')
        ax.set_ylabel('PR-AUC / Average Precision auf Test')
        ax.set_title('Test PR-AUC nach Prefix und Modell')
        ax.legend()
        ax.grid(alpha=0.3)
        save_fig(fig, 'fig_05_test_pr_auc_by_prefix_model.png')
        if len(test_selected):
            fig, ax = plt.subplots(figsize=(10, 5))
            for metric in ['precision', 'recall', 'f1']:
                sub = test_selected[test_selected['model'] == 'logreg_balanced'].sort_values('prefix_len')
                if len(sub):
                    ax.plot(sub['prefix_len'], sub[metric], marker='o', label=f'logreg_{metric}')
            ax.set_xlabel('Prefix-Länge')
            ax.set_ylabel('Score auf Test')
            ax.set_title('Logistic Regression: Precision/Recall/F1 nach Prefix')
            ax.legend()
            ax.grid(alpha=0.3)
            save_fig(fig, 'fig_06_logreg_precision_recall_f1_by_prefix.png')
        best_rows = test_default.sort_values('pr_auc_average_precision', ascending=False)
        best_primary_row = best_rows.iloc[0].to_dict()
        save_json(best_primary_row, '13_best_primary_model_by_test_pr_auc.json')
        print('Bestes Primary-Modell nach Test-PR-AUC:')
        print(best_primary_row)
        logreg_rows = test_default[test_default['model'] == 'logreg_balanced'].sort_values('pr_auc_average_precision', ascending=False)
        if len(logreg_rows) and len(feature_importance_df):
            best_logreg_prefix = int(logreg_rows.iloc[0]['prefix_len'])
            fi = feature_importance_df[(feature_importance_df['model'] == 'logreg_balanced') & (feature_importance_df['prefix_len'] == best_logreg_prefix)].copy()
            if len(fi):
                top_pos = fi.sort_values('importance', ascending=False).head(15)
                top_neg = fi.sort_values('importance', ascending=True).head(15)
                top_combined = pd.concat([top_neg, top_pos], ignore_index=True)
                fig, ax = plt.subplots(figsize=(10, 8))
                plot_df = top_combined.sort_values('importance')
                ax.barh(plot_df['feature'], plot_df['importance'])
                ax.set_xlabel('Logistic Regression Koeffizient')
                ax.set_title(f'Top LogReg-Features, Prefix {best_logreg_prefix}')
                ax.grid(axis='x', alpha=0.3)
                save_fig(fig, 'fig_07_top_logreg_coefficients.png')
        if len(calibration_df) and best_primary_row:
            best_model = best_primary_row['model']
            best_prefix = int(best_primary_row['prefix_len'])
            calib = calibration_df[(calibration_df['model'] == best_model) & (calibration_df['prefix_len'] == best_prefix)].copy()
            if len(calib):
                fig, ax = plt.subplots(figsize=(6, 6))
                ax.plot(calib['mean_predicted_score'], calib['observed_positive_rate'], marker='o')
                ax.plot([0, 1], [0, 1], linestyle='--')
                ax.set_xlabel('Mittlerer vorhergesagter Score')
                ax.set_ylabel('Beobachtete positive Rate')
                ax.set_title(f'Calibration: {best_model}, Prefix {best_prefix}')
                ax.grid(alpha=0.3)
                save_fig(fig, 'fig_08_calibration_best_primary_model.png')
        if len(curve_df) and best_primary_row:
            best_model = best_primary_row['model']
            best_prefix = int(best_primary_row['prefix_len'])
            roc_best = curve_df[(curve_df['model'] == best_model) & (curve_df['prefix_len'] == best_prefix) & (curve_df['curve'] == 'roc')]
            pr_best = curve_df[(curve_df['model'] == best_model) & (curve_df['prefix_len'] == best_prefix) & (curve_df['curve'] == 'pr')]
            if len(roc_best):
                fig, ax = plt.subplots(figsize=(6, 6))
                ax.plot(roc_best['fpr'], roc_best['tpr'])
                ax.plot([0, 1], [0, 1], linestyle='--')
                ax.set_xlabel('False Positive Rate')
                ax.set_ylabel('True Positive Rate')
                ax.set_title(f'ROC Curve: {best_model}, Prefix {best_prefix}')
                ax.grid(alpha=0.3)
                save_fig(fig, 'fig_09_roc_curve_best_primary_model.png')
            if len(pr_best):
                fig, ax = plt.subplots(figsize=(6, 6))
                ax.plot(pr_best['recall'], pr_best['precision'])
                ax.set_xlabel('Recall')
                ax.set_ylabel('Precision')
                ax.set_title(f'Precision-Recall Curve: {best_model}, Prefix {best_prefix}')
                ax.grid(alpha=0.3)
                save_fig(fig, 'fig_10_pr_curve_best_primary_model.png')
cols_show = ['target', 'prefix_len', 'model', 'split', 'threshold_policy', 'roc_auc', 'pr_auc_average_precision', 'precision', 'recall', 'f1', 'balanced_accuracy', 'brier_score', 'prevalence_pct']
if len(primary_eval_df) and len(test_default):
    display(test_default[cols_show].sort_values(['prefix_len', 'pr_auc_average_precision'], ascending=[True, False]))
else:
    display(failure_log_df if 'failure_log_df' in globals() else pd.DataFrame())


In [ ]:
# Robustheitsmodelle
robust_eval_rows = []
robust_feature_importances = []
robust_failure_rows = []
if RUN_LIGHT_ROBUSTNESS_TARGET_MODELS:
    robustness_targets = [lab for lab in ROBUSTNESS_LABELS_FOR_LIGHT_MODELING if lab in case_df.columns]
    print('Robustness targets:', robustness_targets)
    robust_model_df = build_modeling_dataset(ROBUSTNESS_PREFIX_LENGTH)
    for target in robustness_targets:
        if target == PRIMARY_LABEL:
            continue
        if target not in robust_model_df.columns:
            continue
        robust_model_df[target] = robust_to_bool(robust_model_df[target])
        prevalence = float(robust_model_df[target].mean())
        if robust_model_df[target].nunique() < 2:
            msg = f'Robustness target skipped due only one class {target}: prevalence={prevalence:.3f}'
            print(msg)
            analysis_notes.append(msg)
            robust_failure_rows.append({'target': target, 'model': None, 'error': msg})
            continue
        if prevalence < 0.01 or prevalence > 0.8:
            msg = f'Robustness target skipped due prevalence {target}: {prevalence:.3f}'
            print(msg)
            analysis_notes.append(msg)
            robust_failure_rows.append({'target': target, 'model': None, 'error': msg})
            continue
        for model_name in ROBUSTNESS_MODELS:
            print(f'Robustness trainiere: target={target}, prefix={ROBUSTNESS_PREFIX_LENGTH}, model={model_name}')
            try:
                res = fit_and_evaluate(robust_model_df, target, ROBUSTNESS_PREFIX_LENGTH, model_name)
                robust_eval_rows.append(res['eval_df'])
                if len(res['feature_importance_df']):
                    robust_feature_importances.append(res['feature_importance_df'].head(100))
            except Exception as e:
                msg = f'FAILED robustness: target={target}, model={model_name}, error={repr(e)}'
                print(msg)
                analysis_notes.append(msg)
                robust_failure_rows.append({'target': target, 'model': model_name, 'error': repr(e)})
robust_eval_df = pd.concat(robust_eval_rows, ignore_index=True) if robust_eval_rows else pd.DataFrame()
robust_feature_importance_df = pd.concat(robust_feature_importances, ignore_index=True) if robust_feature_importances else pd.DataFrame()
robust_failure_df = pd.DataFrame(robust_failure_rows)
save_csv(robust_eval_df, '14_robustness_target_model_evaluation.csv', index=False)
save_csv(robust_feature_importance_df, '15_robustness_feature_importances_top100_each.csv', index=False)
save_csv(robust_failure_df, '15b_robustness_model_failure_log.csv', index=False)
if len(robust_eval_df):
    robust_test = robust_eval_df[(robust_eval_df['split'] == 'test') & (robust_eval_df['threshold_policy'] == 'fixed_0_5')].copy()
    if len(robust_test):
        fig, ax = plt.subplots(figsize=(10, 6))
        robust_plot = robust_test.sort_values('pr_auc_average_precision')
        y_labels = robust_plot['target'] + ' | ' + robust_plot['model']
        ax.barh(y_labels, robust_plot['pr_auc_average_precision'])
        ax.set_xlabel('Test PR-AUC / Average Precision')
        ax.set_title('Light Robustness Modeling: PR-AUC')
        ax.grid(axis='x', alpha=0.3)
        save_fig(fig, 'fig_11_robustness_targets_pr_auc.png')
robust_eval_df.head()


In [ ]:
# Ergebnisse speichern
if len(primary_eval_df):
    test_rank = primary_eval_df[(primary_eval_df['split'] == 'test') & (primary_eval_df['threshold_policy'] == 'fixed_0_5')].copy()
    test_rank = test_rank.sort_values('pr_auc_average_precision', ascending=False)
    save_csv(test_rank, '16_primary_test_model_ranking_fixed_threshold.csv', index=False)
else:
    test_rank = pd.DataFrame()
if len(primary_eval_df):
    test_rank_selected = primary_eval_df[(primary_eval_df['split'] == 'test') & (primary_eval_df['threshold_policy'] == 'validation_f1_threshold')].copy()
    test_rank_selected = test_rank_selected.sort_values('f1', ascending=False)
    save_csv(test_rank_selected, '17_primary_test_model_ranking_validation_threshold.csv', index=False)
else:
    test_rank_selected = pd.DataFrame()
confusion_cols = ['target', 'prefix_len', 'model', 'split', 'threshold_policy', 'threshold', 'tn', 'fp', 'fn', 'tp', 'precision', 'recall', 'f1']
confusions = primary_eval_df[confusion_cols].copy() if len(primary_eval_df) else pd.DataFrame()
save_csv(confusions, '18_confusion_matrix_metrics_primary.csv', index=False)
